# Imports

In [1]:
import numpy as np

# creating class MLP only using numpy

In [2]:
class MLP_NP:
    def __init__(self, MLP_architecture, learning_rate, momentum, activation_function='sigmoid'):
       
        """
        MLP_architecture: [input_size (dataset features), hidden_layer1_size, hidden_layer2_size, ..., output_size (possible classes)]
        learning_rate: learning rate for weight updates
        momentum: momentum factor for weight updates
        activation_function: choice of activation function for hidden layers (default is 'sigmoid', can also be 'relu') and 'softmax' for output layer
        """
        self.MLP_architecture = MLP_architecture
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.activation_function = activation_function

        """
        weights: list of weight matrices for each layer (including input to first hidden layer and last hidden layer to output layer)
        biases: list of bias vectors for each layer (including first hidden layer and output layer)
        weight_speed: list of matrices to store the previous weight updates for momentum calculation
        bias_speed: list of vectors to store the previous bias updates for momentum calculation
        """
        self.weights = []
        self.biases = []
        self.weight_speed = []
        self.biases_speed = []
       
        # initialization of the weights and biases
        for i in range(len(MLP_architecture) - 1):
       
            # input and output size for the current layer
            input_size = MLP_architecture[i]
            output_size = MLP_architecture[i + 1]
            
            # initialize weights with small random values and biases with zeros
            weight_matrix = np.random.randn(input_size, output_size) * 0.1
            bias_vector = np.zeros((1, output_size))
           
            # store the initialized weights and biases in the respective lists
            self.weights.append(weight_matrix)
            self.biases.append(bias_vector)

            # Initialize speed matrices for momentum
            self.weight_speed.append(np.zeros_like(weight_matrix))
            self.biases_speed.append(np.zeros_like(bias_vector))
    
    """
    Activation functions: Sigmoid, ReLU, softmax, and their derivatives
    their porpouse is to introduce non-linearity into the model, allowing it to learn complex patterns in the data. The derivatives are used during backpropagation to compute gradients for weight updates.
    """
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def sigmoid_derivative(self, x):
        s = self.sigmoid(x)
        return s * (1 - s)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    """
    Forward pass: computes the output of the network for a given input by passing the data through each layer and applying the activation functions.
    """
    def forward_pass(self, X):

        """
        X: input data (shape: [num_samples, input_size])
        layer_inputs: list to store the inputs to each layer (including the original input X) for use in backpropagation
        layer_outputs: list to store the outputs of each layer for use in backpropagation
        """
        self.layer_inputs = []  
        self.layer_outputs = [] 
        
        input_data = X
        for i in range(len(self.MLP_architecture) - 1):

            # Compute the linear transformation (z) for the current layer and store it for backpropagation
            z = np.dot(input_data, self.weights[i]) + self.biases[i]
            self.layer_inputs.append(z)

            # Apply the appropriate activation function based on the layer type and the function specified in the constructor
            if i < len(self.MLP_architecture) - 2:  # Hidden layers
                if self.activation_function == 'sigmoid':
                    output_data = self.sigmoid(z)
                elif self.activation_function == 'relu':
                    output_data = self.relu(z)
                else:
                    raise ValueError("Unsupported activation function for hidden layers.")

            # aplying softmax activation function to the output layer regardless of the specified activation function for hidden layers, as softmax is typically used for multi-class classification problems        
            else:
                output_data = self.softmax(z)

            # Store the output of the current layer
            self.layer_outputs.append(output_data) 
            
            # The output of the current layer becomes the input for the next layer
            input_data = output_data  
        
        # Return the final output of the network
        return input_data  
    
    """
    Backward pass: computes the gradients of the loss with respect to the weights and biases by propagating the error backward through the network using the chain rule of calculus.
    """
    def backward_pass(self, X, y_true):

        # number of samples and predicted output from the forward pass
        m = X.shape[0] 
        y_pred = self.layer_outputs[-1] 

        # Compute the error at the output layer (using cross-entropy + softmax results in a simplified gradient)
        error = y_pred - y_true

        # Backpropagate the error through the layers in reverse order
        for i in reversed(range(len(self.MLP_architecture) - 1)):
            
            # seeing if is the first hidden layer (i=0) to determine the input for gradient calculation
            if i == 0:
                A_prev = X
            else:
                A_prev = self.layer_outputs[i - 1]

            # Calculate the gradients (dW and db) for the current layer
            dW = (1 / m) * np.dot(A_prev.T, error)
            db = (1 / m) * np.sum(error, axis=0, keepdims=True)

            # propagate the error to the previous layer (if not the first layer)
            if i > 0:

                # multiply the current error by the weights of the current layer (transposed) to get the error for the previous layer
                propagation_error = np.dot(error, self.weights[i].T)  
                
                # Apply the derivative of the activation function of the previous layer to the propagated error
                if self.activation_function == 'sigmoid':
                    activation_derivative = self.sigmoid_derivative(self.layer_inputs[i - 1])
                elif self.activation_function == 'relu':
                    activation_derivative = self.relu_derivative(self.layer_inputs[i - 1])
                else:
                    raise ValueError("Unsupported activation function for hidden layers.")
                
                # Update the error variable for the next iteration of the loop to continue backpropagation
                error = propagation_error * activation_derivative
            
            # calculate the momentum updates for weights and biases
            self.weight_speed[i] = self.momentum * self.weight_speed[i] - (self.learning_rate) * dW
            self.biases_speed[i] = self.momentum * self.biases_speed[i] - (self.learning_rate) * db

            # Update the weights and biases using the calculated momentum updates
            self.weights[i] += self.weight_speed[i]
            self.biases[i] += self.biases_speed[i]

# preparing the training function for the MLP

In [3]:
def train(self, X_train, y_train, epochs):
    # Get user input for the architecture of the MLP, learning rate, momentum, and activation function
    
    # Geting the number of hidden layers to construct the architecture of the MLP
    hidden_layer_sizes = []
    number_of_hidden_layers = int(input("Enter the number of hidden layers: "))
    for i in range(number_of_hidden_layers):

        # Get the size of each hidden layer from the user and store it in a list
        hidden_layer_size = int(input(f"Enter the size of hidden layer {i + 1}: "))
        hidden_layer_sizes.append(hidden_layer_size)

    # Creating the architecture of the MLP based on the input size, hidden layer sizes, and output size
    architecture = [X_train.shape[1]] + hidden_layer_sizes + [y_train.shape[1]]
    
    # Get user input for learning rate, momentum, and activation function for hidden layers
    learning_rate = float(input("Enter the learning rate: "))

    momentum = float(input("Enter the momentum factor: "))

    activation_function = input("Enter the activation function for hidden layers (sigmoid or relu): ")

    # Create an instance of the MLP_NP class with the specified architecture, learning rate, momentum, and activation function
    MLP_model = MLP_NP(architecture, learning_rate, momentum, activation_function)
    
    # Train the MLP for the specified number of epochs, performing forward and backward passes and printing the loss at each epoch
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        
        # Perform a forward pass to compute the predicted output
        y_pred = MLP_model.forward_pass(X_train)

        # Perform a backward pass to compute the gradients and update the weights and biases
        MLP_model.backward_pass(X_train, y_train)
 
        # print the loss for the current epoch (using cross-entropy loss)
        loss = np.mean(np.sum(-y_train * np.log(y_pred + 1e-8), axis=1))
        print(f"Epoch Loss: {loss:.4f}")
    
    # After training, return the final predicted output from the forward pass
    y_pred = MLP_model.forward_pass(X_train)
    return y_pred